# MongoDB: One-to-Few Relationships

> **TL;DR** — A one-to-few relationship is best modeled by **embedding** the child data as an array of subdocuments directly inside the parent document.

---

## Overview

| Aspect | Detail |
|---|---|
| **Definition** | One parent record links to a small, bounded number of child items — a person with 2–3 addresses, a user with a few phone numbers, a post with a handful of tags. |
| **Best strategy** | Embed the children in an array field on the parent. |
| **Why it works** | The parent and all its children come back in a **single read** (`find`), with no `$lookup`, no second round trip, and no application-side join. |

```mermaid
graph LR
    A["users document"] --> B["addresses[0] — home"]
    A --> C["addresses[1] — work"]
    A --> D["addresses[2] — billing"]
```

Everything above lives in **one BSON document**, on **one disk page**, retrieved by **one query**.

---

## Example Schema

A `users` collection where each user has a few addresses:

```json
{
  "_id": ObjectId("64a2f8b5f1234567890abcdef"),
  "name": "Jane Doe",
  "email": "jane@example.com",
  "addresses": [
    {
      "type": "home",
      "street": "123 Main St",
      "city": "Springfield",
      "zip": "62701"
    },
    {
      "type": "work",
      "street": "456 Market St",
      "city": "Springfield",
      "zip": "62702"
    }
  ]
}
```

### Mongoose equivalent

```js
const mongoose = require("mongoose");

const addressSchema = new mongoose.Schema(
  {
    type:   { type: String, enum: ["home", "work", "billing"], required: true },
    street: { type: String, required: true },
    city:   { type: String, required: true },
    zip:    { type: String, match: /^\d{5}$/ }
  },
  { _id: false } // drop the auto _id if you don't need to address subdocs individually
);

const userSchema = new mongoose.Schema({
  name:  { type: String, required: true },
  email: { type: String, required: true, unique: true, lowercase: true },
  addresses: {
    type: [addressSchema],
    validate: [arr => arr.length <= 5, "Max 5 addresses per user"]
  }
});

module.exports = mongoose.model("User", userSchema);
```

Two things worth noting:

- `_id: false` on the subschema keeps documents lean. Keep the `_id` if you plan to update a specific subdocument by its own identifier.
- The array `validate` is your own guardrail against unbounded growth. MongoDB won't enforce it for you.

---

## Working With the Embedded Array

### Query by a field inside the array

Dot notation matches if **any** array element satisfies the condition:

```js
db.users.find({ "addresses.city": "Springfield" })
```

### Match multiple conditions on the *same* element

Without `$elemMatch`, the conditions can be satisfied by different elements — a classic bug:

```js
// WRONG: matches a user with a home address in Springfield
// and a *separate* work address in Chicago
db.users.find({ "addresses.type": "work", "addresses.city": "Chicago" })

// RIGHT: both conditions must hold on one element
db.users.find({
  addresses: { $elemMatch: { type: "work", city: "Chicago" } }
})
```

### Add, remove, and update elements

```js
// Append
db.users.updateOne(
  { _id: userId },
  { $push: { addresses: { type: "billing", street: "789 Oak Ave", city: "Springfield", zip: "62703" } } }
)

// Append only if not already present
db.users.updateOne({ _id: userId }, { $addToSet: { addresses: newAddress } })

// Remove by criteria
db.users.updateOne({ _id: userId }, { $pull: { addresses: { type: "billing" } } })

// Update the first matching element (positional $)
db.users.updateOne(
  { _id: userId, "addresses.type": "home" },
  { $set: { "addresses.$.zip": "62799" } }
)

// Update every element matching a condition (arrayFilters)
db.users.updateOne(
  { _id: userId },
  { $set: { "addresses.$[addr].city": "Shelbyville" } },
  { arrayFilters: [{ "addr.city": "Springfield" }] }
)
```

### Return only the matching subdocument

```js
db.users.find(
  { "addresses.type": "work" },
  { name: 1, "addresses.$": 1 }
)
```

### Flatten the array for aggregation

```js
db.users.aggregate([
  { $unwind: "$addresses" },
  { $group: { _id: "$addresses.city", userCount: { $sum: 1 } } }
])
```

### Indexing

An index on an array field is automatically a **multikey index** — one index entry per array element:

```js
db.users.createIndex({ "addresses.zip": 1 })
```

Caveat: a compound index can include at most **one** array field. Indexing two array fields together is rejected.

---

## Key Benefits

- **Fewer queries** — parent and children arrive together in one round trip.
- **Atomic updates** — a write to a single document is atomic in MongoDB, so the parent and its children stay consistent without a transaction.
- **Data locality** — related data sits contiguously on disk, so reads are fast.
- **Simpler code** — no join logic, no orphaned-child cleanup, no referential-integrity checks in the application layer.

---

## When to Use vs. Avoid

### Use embedding when

- The child set is **small and bounded** (roughly under a few hundred, ideally under a few dozen).
- Children are almost always **read together with the parent**.
- Children have **no independent life** — you never query them on their own or share them across parents.
- The child data changes **infrequently** relative to reads.

### Avoid embedding when

- The relationship is **unbounded** — IoT sensor logs, transaction history, comments on a viral post. Documents have a hard **16 MB limit**, and even well under it, huge arrays hurt performance.
- Children are **queried independently** of the parent.
- Children are **shared** between multiple parents (you'd be duplicating data with no single source of truth).
- The array grows steadily over time — repeated document growth causes rewrites and index churn.

For those cases, use **referencing** instead.

---

## The Cardinality Decision Framework

MongoDB's own guidance splits one-to-N into three buckets:

| Pattern | Scale | Model | Example |
|---|---|---|---|
| **One-to-few** | ~2–10s | **Embed** an array of subdocuments in the parent | User → addresses |
| **One-to-many** | ~10s–1000s | **Reference** — parent holds an array of child `ObjectId`s | Product → parts |
| **One-to-squillions** | Unbounded | **Parent reference** — each child stores the parent's `_id` | Host → log messages |

Rule of thumb: **embed by default, and only break out into a separate collection when you have a specific reason to.** The reasons are usually unbounded growth, independent access, or sharing across parents.

### Middle ground: the Subset Pattern

When most reads need only a slice of a large child set, embed the hot subset and reference the rest:

```json
{
  "_id": ObjectId("..."),
  "title": "Understanding MongoDB Schema Design",
  "recentComments": [ /* last 5, embedded for the page render */ ],
  "totalComments": 4821
}
```

Full comment history lives in its own `comments` collection, fetched only when the user clicks "show all."

---

## Coming From SQL

In a relational database this would be two tables and a join:

```sql
SELECT u.name, a.street, a.city
FROM   users u
JOIN   addresses a ON a.user_id = u.id
WHERE  u.id = 42;
```

In MongoDB, embedding collapses that into:

```js
db.users.findOne({ _id: 42 })
```

The trade-off is deliberate. SQL normalizes to avoid duplication and enforce integrity at the schema level. MongoDB denormalizes to match how the application actually reads the data. **Model to your query patterns, not to the entities.**

---

## Common Pitfalls

1. **Unbounded arrays** — the single most common MongoDB anti-pattern. Always ask "what stops this from growing forever?"
2. **Forgetting `$elemMatch`** — multi-condition queries silently match across different elements.
3. **Duplicating shared data** — embedding a category or supplier into every product means updating it in a thousand places.
4. **Ignoring the working set** — large embedded arrays get pulled into RAM on every read, even when you only wanted the parent's name.
5. **Over-indexing subdocument fields** — each multikey index entry is per element, so index size scales with array length.

---

## References

- [MongoDB Docs — Model One-to-Many Relationships with Embedded Documents](https://www.mongodb.com/docs/manual/tutorial/model-embedded-one-to-many-relationships-between-documents/)
- [MongoDB One-to-N Relationships: An Outline](https://javascript.plainenglish.io/mongodb-one-to-n-relationships-an-outline-6cd9e9f9584c)
- [GeeksforGeeks — Create Relationships in MongoDB](https://www.geeksforgeeks.org/mongodb/create-relationship-in-mongodb/)
- [Codefinity — MongoDB 1:Few/Many](https://dev.to/codefinity/mongodb-1-fewmany-29ib)